# Auditoria da necessidade de retreinamento por genoma

Este notebook lê os **splits reais de fragmentos curtos**. Uma inspeção curta mostra comprimentos e alfabetos antes da varredura completa; linhas fora do protocolo de até 512 bp são contabilizadas separadamente, sem interromper a auditoria. Indexa **todo o treino** em SQLite temporário, mede correspondências exatas entre teste e treino (incluindo fita reversa complementar), após remover o padding `_` à direita como no notebook histórico de benchmark e verifica se o teste possui genomas de origem ausentes do treino. Não carrega checkpoints, não treina modelos e não cria dados sintéticos.

**Interpretação:** retreinar só é necessário para responder à pergunta de generalização para *genomas não vistos* quando o checkpoint existente já foi treinado nos genomas disponíveis para teste. Para relatar o experimento histórico no seu escopo de referências conhecidas, é possível manter o checkpoint, mas a sobreposição precisa ser declarada e as alegações limitadas. Um teste sem duplicatas exatas não prova independência genômica.

Se `genome_id` ou `accession` não está associado a cada **fragmento** nos dois splits, o resultado será `PROVENANCE_REQUIRED`: uma lista de genomas obtida posteriormente pelo nome da espécie não estabelece a origem do fragmento.

**Custo:** o modo completo lê aproximadamente 15 milhões de fragmentos de treino e 1,7 milhão de teste; reserve horas e espaço em disco para o índice temporário. `MAX_TEST_ROWS` serve só para verificar o funcionamento inicial e nunca produz uma decisão final.

**Atenção:** uma execução anterior a esta correção pode subestimar correspondências reverso complementares em sequências preenchidas com `_`. Reexecute a auditoria completa e compare o campo `sequence_normalization` no novo `decision.json`. A taxa antiga de 79,98% não é a estimativa final para esse protocolo.


In [1]:
import os
import sys
from pathlib import Path

# Mesmo padrão dos demais notebooks de reprodução: resolve src/ sem instalação editável.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "viral_cnn").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
else:
    raise FileNotFoundError("Abra o notebook a partir do repositório cnn_metagenome")

from viral_cnn.data.retraining_audit import audit_retraining_need, preflight_split, split_files

if os.getenv("SMOKE_TEST", "0").strip().lower() in ("1", "true", "yes"):
    raise RuntimeError("Esta auditoria requer dados reais: remova SMOKE_TEST=1 antes de executar")


## 1. Configure entradas e destino

O padrão procura `cnn_kraken/processed_kraken_species_splited_parquet/{train,test}` dentro de `DATA_ROOT`. Configure `TRAIN_SPLIT_PATH` e `TEST_SPLIT_PATH` para outra topologia. Aponte `AUDIT_WORK_DIR` para um volume com espaço livre suficiente para o índice SQLite temporário; o banco é apagado ao final. O diretório de saída contém apenas o JSON da decisão e a contagem por espécie.


In [2]:
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(PROJECT_ROOT.parent))).expanduser().resolve()
SPLIT_ROOT = DATA_ROOT / "cnn_kraken" / "processed_kraken_species_splited_parquet"
TRAIN_PATH = Path(os.getenv("TRAIN_SPLIT_PATH", str(SPLIT_ROOT / "train"))).expanduser()
TEST_PATH = Path(os.getenv("TEST_SPLIT_PATH", str(SPLIT_ROOT / "test"))).expanduser()
OUTPUT_DIR = Path(os.getenv("AUDIT_OUTPUT_DIR", str(PROJECT_ROOT / "results_reproduction" / "retraining_decision"))).expanduser()
WORK_DIR = os.getenv("AUDIT_WORK_DIR")  # None -> pasta temporária do sistema
MAX_TEST_ROWS = int(os.environ["MAX_TEST_ROWS"]) if os.getenv("MAX_TEST_ROWS") else None

train_files = split_files(TRAIN_PATH)  # Falha se o split real não estiver montado.
test_files = split_files(TEST_PATH)
print(f"Treino: {len(train_files)} arquivo(s); teste: {len(test_files)} arquivo(s)")
print(f"Saída: {OUTPUT_DIR}")
print(f"Índice temporário em: {WORK_DIR or 'diretório temporário do sistema'}")
print("Escopo:", "teste completo" if MAX_TEST_ROWS is None else f"PILOTO, primeiras {MAX_TEST_ROWS} linhas do teste")

# Inspeção rápida: revela comprimentos >512, IUPAC e símbolos inesperados antes do índice completo.
print("Prévia do treino:", preflight_split(TRAIN_PATH, limit=1000))
print("Prévia do teste:", preflight_split(TEST_PATH, limit=1000))


Treino: 256 arquivo(s); teste: 256 arquivo(s)
Saída: /home/mlima/workspace/cnn_metagenome/results_reproduction/retraining_decision
Índice temporário em: diretório temporário do sistema
Escopo: teste completo
Prévia do treino: {'rows_checked': 1000, 'columns': ('sequence_padded', 'label', None), 'sequence_quality': {'benchmark_compatible': 1000}, 'warning': 'First rows only; full audit reports all rows and may differ.'}
Prévia do teste: {'rows_checked': 1000, 'columns': ('sequence_padded', 'label', None), 'sequence_quality': {'benchmark_compatible': 1000}, 'warning': 'First rows only; full audit reports all rows and may differ.'}


## 2. Execute a auditoria somente sobre os dados fornecidos

Uma correspondência exata já demonstra exposição daquela sequência ao treino. Sem identificadores confiáveis de genoma nos fragmentos, o relatório não conclui se há genomas realmente novos no teste. A amostragem parcial não estima a taxa de sobreposição do teste completo.


In [3]:
report = audit_retraining_need(
    TRAIN_PATH, TEST_PATH, OUTPUT_DIR, max_test_rows=MAX_TEST_ROWS, work_dir=WORK_DIR
)
print("Status:", report["status"])
print("Treino:", report["train_rows"], "fragmentos")
print("Teste:", report["test_rows"], "fragmentos")
print("Correspondências exatas/RC:", report["exact_test_matches_train"],
      f'({report["exact_overlap_pct"]:.2f}% do teste auditado)')
print("Proveniência genômica completa:", report["genome_provenance_complete"])
print("Genomas compartilhados:", report["shared_train_test_genomes"])
print("Linhas de genomas não vistos no treino:", report["test_rows_from_unseen_genomes_with_species_in_train"])
print("Recomendação:", report["recommendation"])
print("Arquivos:", OUTPUT_DIR / "decision.json", "e", OUTPUT_DIR / "per_species.csv")

print("Qualidade das sequências no treino:", report["train_sequence_quality"])
print("Qualidade das sequências no teste:", report["test_sequence_quality"])
print("Linhas compatíveis com o benchmark de até 512 bp:", report["benchmark_compatible_test_rows"])
print("Overlap apenas no subconjunto compatível:", report["benchmark_compatible_exact_overlap_pct"])


Status: PROVENANCE_REQUIRED
Treino: 13650160 fragmentos
Teste: 1706270 fragmentos
Correspondências exatas/RC: 1636931 (95.94% do teste auditado)
Proveniência genômica completa: False
Genomas compartilhados: None
Linhas de genomas não vistos no treino: None
Recomendação: Recover trustworthy fragment-level genome IDs before deciding whether an existing checkpoint can be evaluated on unseen genomes.
Arquivos: /home/mlima/workspace/cnn_metagenome/results_reproduction/retraining_decision/decision.json e /home/mlima/workspace/cnn_metagenome/results_reproduction/retraining_decision/per_species.csv
Qualidade das sequências no treino: {'benchmark_compatible': 13649826, 'empty_after_padding': 334}
Qualidade das sequências no teste: {'benchmark_compatible': 1706231, 'empty_after_padding': 39}
Linhas compatíveis com o benchmark de até 512 bp: 1706231
Overlap apenas no subconjunto compatível: 95.9384


## 3. Decisão científica

| Status | Próximo passo |
|---|---|
| `PROVENANCE_REQUIRED` | Recuperar `genome_id`/`accession` **por fragmento** antes de decidir. Não substitua com registros simulados ou um accession consultado pelo nome da espécie. |
| `EVALUATE_UNSEEN_GENOMES_FIRST` | Testar o checkpoint atual no subconjunto de genomas de teste efetivamente ausentes do treino, informando classes cobertas e tamanho do subconjunto. |
| `RETRAIN_FOR_GENOME_GENERALIZATION` | Construir partição agrupada por genoma, retreinar e comparar no **novo** protocolo, se o objetivo exigir generalização para genomas não vistos. |
| `INSUFFICIENT_INDEPENDENT_GENOMES` | Obter referências independentes ou restringir a pergunta científica. |
| `INCONCLUSIVE_PARTIAL_TEST` | Remover `MAX_TEST_ROWS` e repetir o teste completo. |

Se só houver a pergunta sobre **fragmentos de referências conhecidas**, não é obrigatório retreinar para manter o resultado histórico, mas o texto deve declarar a sobreposição e não chamá-lo de generalização para genomas novos. Os resultados deste notebook não testam qualidade preditiva nem indicam que um novo treinamento terá melhor acurácia.
